# OpenStroke: train trên Colab

Chạy lần lượt từng ô. GPU: *Runtime → Change runtime type* → **G4** (High-RAM).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Cài OpenStroke

Luôn lấy bản mới nhất trên GitHub.

In [ ]:
!pip install -q --force-reinstall --no-deps git+https://github.com/Nguyenchitrai62/OpenStroke.git

## 3. Giải nén dataset

Zip giải nén ra `/content/dataset_local/e0` (các file `.npz`, `manifest.json`, `folds.json`).
Đã chốt fold riêng thì chép file của bạn đè lên `/content/dataset_local/e0/folds.json`.

In [ ]:
!unzip -q -o "/content/drive/MyDrive/OpenStroke/openstroke_e0_colab.zip" -d /content/dataset_local


## 4. (Tuỳ chọn) Đo tốc độ và VRAM trước khi train

Dữ liệu giả, khoảng 1 phút. Chọn `batch` lớn nhất có `peak` < 80 GB (G4 có 94 GB); giữ `window=65536`.

In [ ]:
!openstroke bench model=openstroke2 batch=2,4,8 window=65536

## 5. Train

- `model=openstroke2` (mặc định): U-Net transformer kiểu PTv3, cửa sổ 65536 shape (gần cả trang). Lý do và bộ tham số:
  `docs/OPENSTROKE2.md` §4. Không truyền tham số nào thì dùng đúng bộ đó.
- `fold=all`: cross-validation theo PDF (5 fold), hoặc `fold=0,2`.
- Mỗi lần train tạo thư mục mới `runs/train`, `train2`... (`best.pt`, `last.pt`, `results.csv/png` từng fold, `summary.json`).
- Mất phiên: chạy lại ô 1–3 rồi `resume=true` (fold xong được bỏ qua, fold dở chạy tiếp từ `last.pt`).
- Lần đầu tính đặc trưng mất vài phút (cache `runs/cache`).

In [ ]:
!openstroke train \
data=/content/dataset_local/e0 \
model=openstroke2 \
fold=all \
epochs=100 \
batch=4 \
window=65536 \
lr0=0.002

# model=/content/runs/train      (fine-tune: mỗi fold bắt đầu từ fold<k>/best.pt của run đó)
# resume=true                    (chạy tiếp run mới nhất sau khi mất phiên)
# window=16384 batch=16          (hạt giống B của docs/OPENSTROKE2.md: chỉ đổi cửa sổ)
# model=openstroke1 window=4096 batch=8 lr0=0.001 epochs=30   (v1 làm mốc)

## 6. Tải log + `best.pt` của run mới nhất về máy

Gom `train.log`, `summary.json`, `best.pt` của mọi fold thành một zip. Gửi lại file zip này.

In [ ]:
import os
import glob
import shutil
from google.colab import files

train_dirs = [d for d in glob.glob('runs/train*') if os.path.exists(os.path.join(d, 'train.log'))]

if train_dirs:
    latest_train_dir = max(train_dirs, key=os.path.getmtime)
    print(f"Thư mục train mới nhất được phát hiện: {latest_train_dir}")
    shutil.copy2('/content/dataset_local/e0/folds.json', os.path.join(latest_train_dir, 'folds_used.json'))
    tmp = os.path.join('/content/_pack', os.path.basename(latest_train_dir))
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(latest_train_dir, tmp, ignore=shutil.ignore_patterns('last.pt', 'val_scores.npz'))
    zip_path = shutil.make_archive(f'/content/openstroke_{os.path.basename(latest_train_dir)}', 'zip', tmp)
    files.download(zip_path)
    print(f"Đang chuẩn bị tải file: {zip_path} ...")
else:
    print("Không tìm thấy bất kỳ thư mục train nào trong runs/")